# The ***GroupBy*** Pattern

We have seen the ***GroupBy*** operator in ***Pandas***, but  this is actually a more general ***design pattern*** that can be utilized in many data analyics frameworks and data access interfaces, e.g. in ***SQL***.

## GroupBy: general Pattern
<img SRC="IMG/groupby.jpg">

## GroupBy in SQL:
```
SELECT COUNT(CustomerID), Country
FROM Customers
GROUP BY Country
ORDER BY COUNT(CustomerID) DESC;
```

#### Pandas SQL-Query example
```
params=tuple([202101, 202102, 202103])

sql = f""" SELECT p.description, SUM(p.price) as price
          FROM product p
          WHERE p.extenal_id in {params}
          GROUP BY p.description """
result = pd.read_sql_query(sql, con=conn)

```


## GroupBy in Pandas

In [19]:
#setup example
import numpy as np
import pandas as pd
df = pd.DataFrame({'key1' : ['a', 'a', 'b', 'b', 'a'],
                   'key2' : ['one', 'two', 'one', 'two', 'one'],
                   'data1' : np.random.randn(5),
                   'data2' : np.random.randn(5)})
df

,key1,key2,data1,data2
0,a,one,0.273674,0.194519
1,a,two,-0.996968,-0.978965
2,b,one,0.880736,-0.901682
3,b,two,0.511201,-0.410006
4,a,one,-0.398577,0.362973


In [20]:
#group by key1
grouped = df.groupby(df['key1'])
grouped #this is now a more complex group object

In [21]:
#and generates a table per group
for name, group in grouped:
    print ("name:", name, "\n",group)

name: a 
   key1 key2     data1     data2
0    a  one  0.273674  0.194519
1    a  two -0.996968 -0.978965
4    a  one -0.398577  0.362973
name: b 
   key1 key2     data1     data2
2    b  one  0.880736 -0.901682
3    b  two  0.511201 -0.410006


In [22]:
#access group table
grouped.get_group('b')

,key1,key2,data1,data2
2,b,one,0.880736,-0.901682
3,b,two,0.511201,-0.410006


In [23]:
#get numper of entries (rows) per group
grouped.size()

,0
key1,
a,3
b,2


In [24]:
#get number of group entries by columns
grouped.count()

,key2,data1,data2
key1,,,
a,3,3,3
b,2,2,2


#### Think of  grouped DataFrames as 3d objects:

In [25]:
#accessing the "3d" group tables
grouped['data2'].get_group('a')

,data2
0,0.194519
1,-0.978965
4,0.362973


In [26]:
grouped.get_group('a')['data2']

,data2
0,0.194519
1,-0.978965
4,0.362973


### Group by external keys

In [27]:
#define external key years as numpy array
years = np.array([2005, 2005, 2006, 2005, 2006])
df['data1'].groupby([years]).mean()


,data1
2005,-0.070698
2006,0.241080


### Group by functions

In [28]:
#sort by column and retun top n
def top(df, n=5, column='data1'):
    return df.sort_values(by=column)[-n:]

df.groupby(df['key1']).apply(top,n=5)

/tmp/ipykernel_498/1087598677.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby(df['key1']).apply(top,n=5)


key1 key2     data1     data2
key1                                
a    1    a  two -0.996968 -0.978965
     4    a  one -0.398577  0.362973
     0    a  one  0.273674  0.194519
b    3    b  two  0.511201 -0.410006
     2    b  one  0.880736 -0.901682

### Group-wise aggregation (apply)
<img SRC="IMG/groupby.jpg">

#### Typical build in aggregation functions:
* sum
* mean
* max / min
* quantile
* ...

In [29]:
#aggregate over the groups
grouped.count()

,key2,data1,data2
key1,,,
a,3,3,3
b,2,2,2


In [30]:
grouped.sum()

,key2,data1,data2
key1,,,
a,onetwoone,-1.121871,-0.421472
b,onetwo,1.391937,-1.311688


#### Custom Aggregation Functions

In [31]:
def peak_to_peak(arr):
    return arr.max() - arr.min()
grouped.agg({'data1':peak_to_peak})

,data1
key1,
a,1.270642
b,0.369536


#### Multiple aggregations

In [32]:
#just call a list of function
grouped[['data1']].agg([peak_to_peak, 'mean', 'median'])

data1                    
     peak_to_peak      mean    median
key1                                 
a        1.270642 -0.373957 -0.398577
b        0.369536  0.695968  0.695968

#### Suppressing the Group Keys

In [33]:
df.groupby(df['key1']).apply(top,n=2)

/tmp/ipykernel_498/3741689088.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby(df['key1']).apply(top,n=2)


key1 key2     data1     data2
key1                                
a    4    a  one -0.398577  0.362973
     0    a  one  0.273674  0.194519
b    3    b  two  0.511201 -0.410006
     2    b  one  0.880736 -0.901682

In [34]:
df.groupby(df['key1'], group_keys=False).apply(top,n=2)

/tmp/ipykernel_498/3060087978.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby(df['key1'], group_keys=False).apply(top,n=2)


,key1,key2,data1,data2
4,a,one,-0.398577,0.362973
0,a,one,0.273674,0.194519
3,b,two,0.511201,-0.410006
2,b,one,0.880736,-0.901682


## More  Exercises in the Lab session...